# Week 4 — Supervised Learning: Titanic Survival Prediction

**Goal:** Predict whether a Titanic passenger survived using supervised learning.

This notebook compares Logistic Regression and Decision Tree. Run cells from top to bottom. The metrics should be close to those documented in the report; results can vary if the dataset or code changes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")


## 1. Load and inspect the dataset

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values by column:")
print(df.isnull().sum())


## 2. Select features and target

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features].copy()
y = df['Survived'].copy()

# This matches the original experiment used for the report.
# Note: this imputation happens before splitting and can cause minor data leakage.
X['Age'] = X['Age'].fillna(X['Age'].median())
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


## 3. Build preprocessing pipeline

In [ ]:
numerical_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])


## 4. Split into training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 5. Train and evaluate Logistic Regression

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

log_accuracy = accuracy_score(y_test, y_pred)
log_precision = precision_score(y_test, y_pred)
log_recall = recall_score(y_test, y_pred)
log_f1 = f1_score(y_test, y_pred)

print(f"Test accuracy: {log_accuracy:.4f} ({log_accuracy:.2%})")
print(f"Precision:     {log_precision:.4f} ({log_precision:.2%})")
print(f"Recall:        {log_recall:.4f} ({log_recall:.2%})")
print(f"F1-score:      {log_f1:.4f} ({log_f1:.2%})")
print("\\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


## 6. Five-fold stratified cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')

print("Fold accuracies:", [f"{score:.2%}" for score in cv_scores])
print(f"Mean CV accuracy: {cv_scores.mean():.2%}")
print(f"Standard deviation: {cv_scores.std():.4f}")


## 7. Train and evaluate Decision Tree

In [ ]:
tree_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))
])

tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_precision = precision_score(y_test, tree_pred)
tree_recall = recall_score(y_test, tree_pred)
tree_f1 = f1_score(y_test, tree_pred)

tree_cv_scores = cross_val_score(tree_model, X_train, y_train, cv=cv, scoring='accuracy')

print(f"Test accuracy: {tree_accuracy:.4f} ({tree_accuracy:.2%})")
print(f"Precision:     {tree_precision:.4f} ({tree_precision:.2%})")
print(f"Recall:        {tree_recall:.4f} ({tree_recall:.2%})")
print(f"F1-score:      {tree_f1:.4f} ({tree_f1:.2%})")
print(f"Mean CV accuracy: {tree_cv_scores.mean():.2%}")


## 8. Compare models

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Precision', 'Recall', 'F1-score', 'Mean 5-fold CV accuracy'],
    'Logistic Regression': [
        log_accuracy, log_precision, log_recall, log_f1, cv_scores.mean()
    ],
    'Decision Tree': [
        tree_accuracy, tree_precision, tree_recall, tree_f1, tree_cv_scores.mean()
    ]
})
display(comparison.style.format({
    'Logistic Regression': '{:.2%}',
    'Decision Tree': '{:.2%}'
}))


## 9. Visualize test and cross-validation accuracy

In [ ]:
models = ['Logistic Regression', 'Decision Tree']
test_accuracy = [log_accuracy * 100, tree_accuracy * 100]
cv_accuracy = [cv_scores.mean() * 100, tree_cv_scores.mean() * 100]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, test_accuracy, width, label='Test Accuracy')
plt.bar(x + width/2, cv_accuracy, width, label='Cross-Validation Accuracy')
plt.ylabel('Accuracy (%)')
plt.xlabel('Machine Learning Models')
plt.title('Performance Comparison of Supervised Learning Models')
plt.xticks(x, models)
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()


## 10. Conclusion

Logistic Regression performed better on the held-out test set for accuracy, precision, recall, and F1-score. The Decision Tree had a slightly higher mean cross-validation accuracy. The results show why multiple metrics and validation are useful.

**Limitation:** The original experiment filled `Age` and `Embarked` before splitting the data. For best practice, avoid this by removing those two `fillna` lines and letting the pipeline perform imputation after the split. If you make this change, rerun all cells and update the report metrics because they may change.
